In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "validation").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from validation.notebook_bootstrap import bootstrap, workshop_retrieved_references

bedrock_model_arn, load_workshop_state, persist_workshop_state_file = bootstrap()


# Query Reformulation com suporte do Amazon Bedrock Knowledge Bases

Otimizar qualidade, custo e latência são alguns dos fatores mais importantes ao desenvolver aplicações GenAI baseadas em RAG. Muitas vezes, as queries de entrada para um Foundation Model (FM) podem ser muito complexas, com múltiplas perguntas e relações complexas. Com queries tão complexas, a etapa de embedding pode mascarar ou diluir componentes importantes da query, resultando em chunks recuperados que podem não fornecer contexto para todos os aspectos da query. Isso pode produzir uma resposta abaixo do desejável na sua aplicação RAG.

Agora, com query reformulation, podemos pegar um prompt de entrada complexo e dividi-lo em múltiplas sub-queries. Essas sub-queries passarão separadamente por suas próprias etapas de retrieval para chunks relevantes. Os chunks resultantes serão então agrupados e ranqueados juntos antes de serem passados ao FM para gerar uma resposta. Query reformulation é mais uma ferramenta que podemos usar para ajudar a aumentar a acurácia em queries complexas que sua aplicação pode enfrentar em produção.

# Setup do Notebook
Siga os passos abaixo com uma role compatível e um ambiente de computação para começar

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --no-deps --quiet
%pip install -r ../requirements.txt --upgrade --quiet

In [ ]:
# Kernel restart is intentionally skipped in corrected notebooks.


In [ ]:
import os
import sys
import time
import boto3
import logging
import pprint
import json

# Set the path to import module
from pathlib import Path
current_path = Path().resolve()
current_path = current_path.parent
if str(current_path) not in sys.path:
    sys.path.append(str(current_path))
# Print sys.path to verify
# print(sys.path)

from utils.knowledge_base import BedrockKnowledgeBase

In [ ]:
#Clients
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region =  session.region_name
account_id = sts_client.get_caller_identity()["Account"]
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime') 
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)
region, account_id

In [ ]:
import time

# Get the current timestamp
current_time = time.time()

# Format the timestamp as a string
timestamp_str = time.strftime("%Y%m%d%H%M%S", time.localtime(current_time))[-7:]
# Create the suffix using the timestamp
suffix = f"{timestamp_str}"
knowledge_base_name_standard = 'standard-kb'
knowledge_base_description = "Octank 10k KB"
bucket_name = f'{knowledge_base_name_standard}-{suffix}'

foundation_model = os.getenv("BEDROCK_TEXT_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")

data_source=[{"type": "S3", "bucket_name": bucket_name}]

## 2 - Criar knowledge bases com estratégia de fixed chunking

In [ ]:
knowledge_base_standard = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name_standard}-{suffix}',
    kb_description=knowledge_base_description,
    data_sources=data_source,
    chunking_strategy = "FIXED_SIZE", 
    suffix = f'{suffix}-f'
)

## 2.1 Fazer upload do dataset para o Amazon S3
Agora que criamos a knowledge base, vamos populá-la com o dataset do relatório `Octank financial 10K`. A data source da Knowledge Base espera que os dados estejam disponíveis no bucket S3 conectado a ela, e as alterações nos dados podem ser sincronizadas com a knowledge base usando a chamada de API `StartIngestionJob`. Neste exemplo, usaremos a [abstração boto3](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent/client/start_ingestion_job.html) da API, via nossa classe helper.

Vamos primeiro fazer o upload dos dados disponíveis na pasta `dataset` para o S3.

In [ ]:
import os

def upload_directory(path, bucket_name):
    for root, dirs, files in os.walk(path):
        for file in files:
            file_to_upload = os.path.join(root, file)
            if file not in ["LICENSE", "NOTICE", "README.md"]:
                print(f"uploading file {file_to_upload} to {bucket_name}")
                s3_client.upload_file(file_to_upload, bucket_name, file)
            else:
                print(f"Skipping file {file_to_upload}")

upload_directory("../synthetic_dataset", bucket_name)


In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_standard.start_ingestion_job()

In [ ]:
kb_id = knowledge_base_standard.get_knowledge_base_id()

# Query Reformulation em Ação

Neste notebook, investigaremos uma query simples e uma mais complexa que poderiam se beneficiar de query reformulation e veremos como isso afeta as respostas geradas.

##  Prompt complexo

Para demonstrar a funcionalidade, vamos analisar uma query que faz diversas perguntas sobre informações contidas no documento financeiro Octank 10K. Esta query contém vários pedidos que não estão semanticamente relacionados. Quando essa query é transformada em embedding durante a etapa de retrieval, alguns aspectos da query podem ser diluídos e, portanto, os chunks relevantes retornados podem não abordar todos os componentes desta query complexa.

Para consultar nossa Knowledge Base e gerar uma resposta, usaremos a chamada de API __retrieve_and_generate__. Para usar o recurso de query reformulation, incluiremos na configuração da knowledge base as informações adicionais mostradas abaixo:

```
'orchestrationConfiguration': {
        'queryTransformationConfiguration': {
            'type': 'QUERY_DECOMPOSITION'
        }
    }
```

__Nota:__ A estrutura da resposta de saída é a mesma de um __retrieve_and_generate__ normal sem query reformulation.

#### Sem Query Reformulation

Vamos ver como o resultado gerado fica para a seguinte query sem usar query reformulation:

"Where is the Octank company waterfront building located and how does the whistleblower scandal hurt the company and its image?"

In [ ]:
query = "What is octank tower and how does the whistleblower scandal hurt the company and its image?"

In [ ]:
response_ret = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)


# generated text output

print(response_ret['output']['text'],end='\n'*2)

In [ ]:
response_without_qr = workshop_retrieved_references(response_ret)
print("# of citations or chunks used to generate the response: ", len(response_without_qr))
def citations_rag_print(response_ret):
#structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
    for num,chunk in enumerate(response_ret,1):
        print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
        print(f'Chunk {num} Location: ',chunk['location'],end='\n'*2)
        print(f'Chunk {num} Metadata: ',chunk['metadata'],end='\n'*2)

citations_rag_print(response_without_qr)

Como visto nas citações acima, nosso retrieval com a query complexa não retornou nenhum chunk relevante sobre o prédio, focando em embeddings mais similares ao incidente do whistleblower.

Isso pode indicar que o embedding da query resultou em alguma diluição da semântica daquela parte da query.

#### Com Query Reformulation

Agora vamos ver como query reformulation pode beneficiar o retrieval de contexto mais alinhado, o que, por sua vez, vai melhorar a acurácia da geração de respostas.

In [ ]:
response_ret = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            },
            'orchestrationConfiguration': {
                'queryTransformationConfiguration': {
                    'type': 'QUERY_DECOMPOSITION'
                }
            }
        }
    }
)


# generated text output

print(response_ret['output']['text'],end='\n'*2)

Vamos analisar os chunks recuperados com query reformulation

In [ ]:
response_with_qr = workshop_retrieved_references(response_ret)
print("# of citations or chunks used to generate the response: ", len(response_with_qr))


citations_rag_print(response_with_qr)

Podemos ver que, com query reformulation ativado, os chunks recuperados agora fornecem contexto tanto para o escândalo do whistleblower quanto para a localização da propriedade waterfront.

### Observando a decomposição do prompt usando CloudWatch Logs

Antes de realizar o retrieval, a query complexa é decomposta em múltiplas sub-queries. Isso pode ser visto no exemplo de query acima quando isolamos a invocação para a ação de decomposição, onde nossa __standalone_question__ é a query original e as sub-queries resultantes são mostradas entre tags __\<query\>__

__Nota__: Você deve habilitar o invocation logging no Bedrock para que os logs possam ser visualizados no CloudWatch. Consulte [aqui](https://docs.aws.amazon.com/bedrock/latest/userguide/model-invocation-logging.html) para detalhes.


```
<generated_queries>

<standalone_question>
What is octank tower and how does the whistleblower scandal hurt the company and its image?
</standalone_question>

<query>
What is octank tower?
</query>

<query>
What is the whistleblower scandal involving Octank company?
</query>

<query>
How did the whistleblower scandal affect Octank company's reputation and public image?
</query>

</generated_queries>
```

<div class="alert alert-block alert-warning">
<b>Nota:</b> Lembre-se de excluir a KB, o índice OSS e as roles e policies IAM relacionadas para evitar a cobrança de custos.
</div>

In [ ]:
# Cleanup is intentionally deferred to full_cleanup.ipynb.
print("Cleanup deferred to full_cleanup.ipynb.")


Agora que vimos como query reformulation funciona e como pode melhorar respostas a queries complexas, convidamos você a se aprofundar e experimentar com esta técnica para otimizar seu workflow de RAG.